# Q4 - Real-world Dataset 1 (bread_basket.csv)
ทดลองหลายค่า support/confidence และวิเคราะห์ Spurious + Surprising patterns

In [1]:
import warnings

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 150)

In [2]:
raw = pd.read_csv("dataset/bread_basket.csv")
print("Raw rows:", len(raw))
print("Unique transactions:", raw["Transaction"].nunique())
print("Unique items:", raw["Item"].nunique())
print("\nTop 15 items:")
print(raw["Item"].value_counts().head(15))

basket = (
    raw.groupby(["Transaction", "Item"])["Item"]
    .count()
    .unstack(fill_value=0)
    .gt(0)
)
print("\nBasket shape:", basket.shape)

Raw rows: 20507
Unique transactions: 9465
Unique items: 94

Top 15 items:
Item
Coffee           5471
Bread            3325
Tea              1435
Cake             1025
Pastry            856
Sandwich          771
Medialuna         616
Hot chocolate     590
Cookies           540
Brownie           379
Farm House        374
Muffin            370
Juice             369
Alfajores         369
Soup              342
Name: count, dtype: int64

Basket shape: (9465, 94)


In [3]:
support_grid = [0.01, 0.02, 0.03, 0.05]
support_scan_rows = []
for ms in support_grid:
    fi = apriori(basket, min_support=ms, use_colnames=True, max_len=5)
    support_scan_rows.append([ms, len(fi), int((fi["itemsets"].apply(len) >= 2).sum())])

support_scan = pd.DataFrame(
    support_scan_rows,
    columns=["min_support", "all_frequent_itemsets", "frequent_itemsets_ge2"],
)
print("Support scan:")
print(support_scan.to_string(index=False))

Support scan:
 min_support  all_frequent_itemsets  frequent_itemsets_ge2
        0.01                     61                     31
        0.02                     33                     14
        0.03                     23                      6
        0.05                     11                      2


In [4]:
min_sup = 0.02
min_conf = 0.30

fi = apriori(basket, min_support=min_sup, use_colnames=True, max_len=5)
rules = association_rules(
    fi,
    metric="support",
    min_threshold=min_sup,
    num_itemsets=len(fi),
)

rules_A = rules[(rules["support"] >= min_sup) & (rules["confidence"] >= min_conf)].copy()
rules_B = rules_A[rules_A["lift"] > 1].copy()

def p_value_for_rule(row):
    antecedent = list(row["antecedents"])
    consequent = list(row["consequents"])
    x = basket[antecedent].all(axis=1)
    y = basket[consequent].all(axis=1)
    n11 = int(((x) & (y)).sum())
    n10 = int(((x) & (~y)).sum())
    n01 = int(((~x) & (y)).sum())
    n00 = int(((~x) & (~y)).sum())

    row1 = n11 + n10
    row2 = n01 + n00
    col1 = n11 + n01
    col2 = n10 + n00
    if row1 == 0 or row2 == 0 or col1 == 0 or col2 == 0:
        return 1.0

    _, p_value, _, _ = chi2_contingency([[n11, n10], [n01, n00]])
    return float(p_value)

if not rules_B.empty:
    rules_B["p_value"] = rules_B.apply(p_value_for_rule, axis=1)
else:
    rules_B["p_value"] = pd.Series(dtype=float)

rules_C = rules_B[rules_B["p_value"] < 0.05].copy()
removed_B = rules_A[rules_A["lift"] <= 1].copy()
removed_C = rules_B[rules_B["p_value"] >= 0.05].copy()

print(f"Chosen min_support={min_sup}, min_confidence={min_conf}")
print("All rules:", len(rules))
print("(A) Basic rules:", len(rules_A))
print("(B) Strong rules:", len(rules_B))
print("(C) Significant rules:", len(rules_C))

print("\nTop significant rules:")
if not rules_C.empty:
    cols = ["antecedents", "consequents", "support", "confidence", "lift", "p_value"]
    print(rules_C.sort_values("lift", ascending=False).head(15)[cols].to_string(index=False))
else:
    print("No significant rules.")

print("\nExamples of rules removed in B (lift <= 1):")
if not removed_B.empty:
    print(removed_B[["antecedents", "consequents", "support", "confidence", "lift"]].head(10).to_string(index=False))
else:
    print("No rules removed in B.")

print("\nExamples of rules removed in C (p-value >= 0.05):")
if not removed_C.empty:
    print(removed_C[["antecedents", "consequents", "support", "confidence", "lift", "p_value"]].head(10).to_string(index=False))
else:
    print("No rules removed in C.")

Chosen min_support=0.02, min_confidence=0.3
All rules: 28
(A) Basic rules: 10
(B) Strong rules: 9
(C) Significant rules: 6

Top significant rules:
           antecedents         consequents  support  confidence     lift      p_value
    frozenset({Toast}) frozenset({Coffee}) 0.023666    0.704403 1.472431 3.636144e-16
frozenset({Medialuna}) frozenset({Coffee}) 0.035182    0.569231 1.189878 6.858326e-06
   frozenset({Pastry}) frozenset({Coffee}) 0.047544    0.552147 1.154168 1.228884e-05
    frozenset({Juice}) frozenset({Coffee}) 0.020602    0.534247 1.116750 3.357804e-02
 frozenset({Sandwich}) frozenset({Coffee}) 0.038246    0.532353 1.112792 3.927546e-03
     frozenset({Cake}) frozenset({Coffee}) 0.054728    0.526958 1.101515 1.441754e-03

Examples of rules removed in B (lift <= 1):
     antecedents         consequents  support  confidence    lift
frozenset({Tea}) frozenset({Coffee}) 0.049868     0.34963 0.73084

Examples of rules removed in C (p-value >= 0.05):
               antecede